In [9]:
# ============================================================================
#          FRAUD DETECTION - VERSION 7.0 COMPLETE (FULLY OPTIMIZED)
#          + Advanced Feature Engineering (34 features)
#          + Deep Network Architecture [256, 128, 64, 32]
#          + Aggressive Hyperparameters for Maximum Sensitivity
# ============================================================================

import Pkg
Pkg.add(["CSV", "DataFrames", "Flux", "Statistics", "Random", "Dates", "StatsBase"])

using CSV
using DataFrames
using Flux
using Flux.Losses
using Statistics
using Random
using Dates
using StatsBase

println("✅ Packages loaded!")

# ============================================================================
#              UTILITY FUNCTIONS
# ============================================================================

function oneHotEncoding(feature::AbstractArray{<:Any,1}, classes::AbstractArray{<:Any,1})
    @assert(all([in(value, classes) for value in feature]))
    numClasses = length(classes)
    if (numClasses == 2)
        return reshape(feature .== classes[2], :, 1)
    else
        oneHot = falses(length(feature), numClasses)
        for i in 1:numClasses
            oneHot[:, i] .= (feature .== classes[i])
        end
        return oneHot
    end
end

oneHotEncoding(feature::AbstractArray{<:Any,1}) = oneHotEncoding(feature, sort(unique(feature)))
oneHotEncoding(feature::AbstractArray{Bool,1}) = reshape(feature, :, 1)

calculateMinMaxNormalizationParameters(dataset::AbstractArray{<:Real,2}) = 
    (minimum(dataset, dims=1), maximum(dataset, dims=1))

function normalizeMinMax!(dataset::AbstractArray{<:Real,2}, 
                          normalizationParameters::NTuple{2, AbstractArray{<:Real,2}})
    minValues, maxValues = normalizationParameters
    dataset .-= minValues
    range_vals = maxValues .- minValues
    for j in 1:size(dataset, 2)
        if range_vals[j] > 0
            dataset[:, j] ./= range_vals[j]
        else
            dataset[:, j] .= 0
        end
    end
    return dataset
end

normalizeMinMax(dataset::AbstractArray{<:Real,2}, normalizationParameters::NTuple{2, AbstractArray{<:Real,2}}) = 
    normalizeMinMax!(copy(Float32.(dataset)), normalizationParameters)

# --- Metrics ---
function classifyOutputs(outputs::AbstractArray{<:Real,2}; threshold::Real=0.5)
    return outputs .>= threshold
end

accuracy(outputs::AbstractArray{Bool,1}, targets::AbstractArray{Bool,1}) = mean(outputs .== targets)

function confusionMatrix(outputs::AbstractArray{Bool,1}, targets::AbstractArray{Bool,1})
    TP = sum(outputs .& targets)
    TN = sum(.!outputs .& .!targets)
    FP = sum(outputs .& .!targets)
    FN = sum(.!outputs .& targets)
    
    total = TP + TN + FP + FN
    acc = total > 0 ? (TP + TN) / total : 0.0
    err = 1.0 - acc
    sens = (TP + FN) > 0 ? TP / (TP + FN) : 0.0
    spec = (TN + FP) > 0 ? TN / (TN + FP) : 0.0
    ppv  = (TP + FP) > 0 ? TP / (TP + FP) : 0.0
    npv  = (TN + FN) > 0 ? TN / (TN + FN) : 0.0
    f1 = (sens + ppv) > 0 ? 2 * sens * ppv / (sens + ppv) : 0.0
    
    cm = Float64.([TN FP; FN TP])
    return acc, err, sens, spec, ppv, npv, f1, cm
end

# --- ANN BUILDER ---
function buildClassANN(numInputs::Int, topology::AbstractArray{<:Int,1}, numOutputs::Int;
                       transferFunctions::AbstractArray{<:Function,1}=fill(relu, length(topology)))
    ann = Chain()
    input_size = numInputs
    for (i, num_neurons) in enumerate(topology)
        ann = Chain(ann..., Dense(input_size, num_neurons, transferFunctions[i]))
        input_size = num_neurons
    end
    ann = Chain(ann..., Dense(input_size, 1, σ)) 
    return ann
end

# --- TRAINER ---
function trainClassANNWeighted(topology::AbstractArray{<:Int,1},
                      trainingDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}};
                      validationDataset::Union{Nothing, Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}}}=nothing,
                      transferFunctions::AbstractArray{<:Function,1}=fill(relu, length(topology)),
                      maxEpochs::Int=1000, minLoss::Real=0.0, learningRate::Real=0.01,
                      maxEpochsVal::Int=20, 
                      posWeight::Real=1.0)
    
    (trainInputs, trainTargets) = trainingDataset
    ann = buildClassANN(size(trainInputs,2), topology, size(trainTargets,2); transferFunctions=transferFunctions)
    
    function weighted_bce(ŷ, y)
        ŷ_safe = clamp.(ŷ, 1e-7, 1 - 1e-7)
        class_weight = y .* posWeight .+ (1 .- y) .* 1.0
        bce = -(y .* log.(ŷ_safe) .+ (1 .- y) .* log.(1 .- ŷ_safe))
        return mean(class_weight .* bce)
    end
    
    loss_fn(m, x, y) = weighted_bce(m(x), y)
    opt_state = Flux.setup(Adam(learningRate), ann)
    
    bestAnn = deepcopy(ann)
    bestValLoss = Inf
    epochsSinceImprovement = 0
    
    for epoch in 1:maxEpochs
        Flux.train!(loss_fn, ann, [(trainInputs', trainTargets')], opt_state)
        
        if validationDataset !== nothing
            (valInputs, valTargets) = validationDataset
            curr_val_loss = loss_fn(ann, valInputs', valTargets')
            
            if curr_val_loss < bestValLoss
                bestValLoss = curr_val_loss
                bestAnn = deepcopy(ann)
                epochsSinceImprovement = 0
            else
                epochsSinceImprovement += 1
            end
            
            if epochsSinceImprovement >= maxEpochsVal
                return bestAnn
            end
        end
    end
    return validationDataset !== nothing ? bestAnn : ann
end

# --- Cross-Validation Utility ---
function crossvalidation(targets::AbstractArray{Bool,1}, k::Int64)
    indices = zeros(Int, length(targets))
    indices[targets] .= crossvalidation_random(sum(targets), k)
    indices[.!targets] .= crossvalidation_random(sum(.!targets), k)
    return indices
end

function crossvalidation_random(N::Int64, k::Int64)
    base = collect(1:k)
    repeated = repeat(base, Int(ceil(N/k)))
    idx = repeated[1:N]
    shuffle!(idx)
    return idx
end

# --- CV MAIN FUNCTION ---
function ANNCrossValidationBigData(topology::AbstractArray{<:Int,1},
        dataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,1}},
        crossValidationIndices::Array{Int64,1};
        numExecutions::Int=1,
        transferFunctions::AbstractArray{<:Function,1}=fill(relu, length(topology)),
        maxEpochs::Int=1000, learningRate::Real=0.01,
        maxEpochsVal::Int=15, 
        threshold::Real=0.5,
        posWeight::Real=1.0)
    
    inputs, targets = dataset
    encoded_targets = reshape(targets, :, 1)
    numFolds = maximum(crossValidationIndices)
    
    fold_metrics = zeros(numFolds, 7)
    globalCM = zeros(Float64, 2, 2)
    
    println("Params -> Threshold: $threshold | PosWeight: $(round(posWeight, digits=1))")
    
    for fold in 1:numFolds
        println("\n--- Fold $fold/$numFolds ---")
        
        testMask = crossValidationIndices .== fold
        trainMask = .!testMask 
        
        train_indices_all = findall(trainMask)
        n_val = floor(Int, length(train_indices_all) * 0.10)
        shuffle!(train_indices_all) 
        
        val_idx = train_indices_all[1:n_val]
        actual_train_idx = train_indices_all[n_val+1:end]
        
        rawTrainInputs = inputs[actual_train_idx, :]
        rawTrainTargets = encoded_targets[actual_train_idx, :]
        
        rawValInputs = inputs[val_idx, :]
        rawValTargets = encoded_targets[val_idx, :]
        
        rawTestInputs = inputs[testMask, :]
        rawTestTargets = encoded_targets[testMask, :]
        
        normParams = calculateMinMaxNormalizationParameters(rawTrainInputs)
        trainInputsNorm = normalizeMinMax(rawTrainInputs, normParams)
        valInputsNorm   = normalizeMinMax(rawValInputs, normParams)
        testInputsNorm  = normalizeMinMax(rawTestInputs, normParams)

        exec_metrics = zeros(numExecutions, 7)
        exec_CMs = zeros(Float64, 2, 2, numExecutions)

        for exec in 1:numExecutions
            ann = trainClassANNWeighted(topology, 
                                    (trainInputsNorm, Bool.(rawTrainTargets));
                                    validationDataset = (valInputsNorm, Bool.(rawValTargets)),
                                    transferFunctions=transferFunctions,
                                    maxEpochs=maxEpochs, 
                                    learningRate=learningRate,
                                    posWeight=posWeight,
                                    maxEpochsVal=maxEpochsVal)
            
            testOutputs = ann(testInputsNorm')'
            testPredictions = testOutputs .>= threshold
            
            acc, err, sens, spec, ppv, npv, f1, cm = confusionMatrix(vec(testPredictions), vec(Bool.(rawTestTargets)))
            exec_metrics[exec, :] = [acc, err, sens, spec, ppv, npv, f1]
            exec_CMs[:, :, exec] = cm
        end
        
        fold_metrics[fold, :] = mean(exec_metrics, dims=1)
        globalCM += mean(exec_CMs, dims=3)[:,:,1]
        
        println("  → Sens: $(round(fold_metrics[fold, 3]*100, digits=1))% | Prec: $(round(fold_metrics[fold, 5]*100, digits=1))% | F1: $(round(fold_metrics[fold, 7]*100, digits=1))%")
    end
    
    means = mean(fold_metrics, dims=1)
    stds = std(fold_metrics, dims=1)
    
    return (means[1], stds[1]), (means[2], stds[2]), (means[3], stds[3]), 
           (means[4], stds[4]), (means[5], stds[5]), (means[6], stds[6]), 
           (means[7], stds[7]), globalCM
end

println("✅ Updated Utility functions!")

# ============================================================================
#              DATA LOADING
# ============================================================================

const DATA_PATH = "Fraudulent_E-Commerce_Transaction_Data_merge.csv"
println("\nLoading full dataset...")
df = CSV.read(DATA_PATH, DataFrame)

target_col = "Is Fraudulent"

fraud_rows = df[df[:, target_col] .== 1, :]
n_fraud_total = size(fraud_rows, 1)
println("Found Total Frauds: $n_fraud_total")

non_fraud_rows = df[df[:, target_col] .== 0, :]
non_fraud_indices = shuffle(1:size(non_fraud_rows, 1))[1:n_fraud_total]
non_fraud_sample = non_fraud_rows[non_fraud_indices, :]

df_balanced = vcat(fraud_rows, non_fraud_sample)
df_balanced = df_balanced[shuffle(1:size(df_balanced, 1)), :] 

println("Balanced Dataset Size: $(size(df_balanced)) (50% Fraud / 50% Legit)")

# ============================================================================
#              ENHANCED PREPROCESSING WITH FEATURE ENGINEERING
# ============================================================================

function preprocess_data_enhanced(dataframe)
    data = copy(dataframe)
    
    println("\n🔧 Creating Enhanced Features...")
    
    # ========== TIME FEATURES (BASE) ==========
    if "Transaction Date" in names(data)
        try
            if eltype(data[!, "Transaction Date"]) <: AbstractString
                 data.ParsedDate = DateTime.(data[!, "Transaction Date"], dateformat"y-m-d H:M:S") 
            else
                 data.ParsedDate = data[!, "Transaction Date"]
            end
            data.Hour = hour.(data.ParsedDate)
            data.Is_Night = [h < 6 ? 1.0 : 0.0 for h in data.Hour]
            
            # ========== NEW: ADVANCED TIME FEATURES ==========
            data.Is_Weekend = [dayofweek(d) in [6,7] ? 1.0 : 0.0 for d in data.ParsedDate]
            data.Hour_Risk = [h in [0,1,2,3,4,5,23] ? 1.0 : 0.0 for h in data.Hour]
            data.Is_Early_Morning = [h in [2,3,4,5] ? 1.0 : 0.0 for h in data.Hour]
            
            println("  ✓ Time features created")
        catch e
            println("  ⚠ Error processing dates: $e")
        end
    end

    # ========== IMPUTATION ==========
    for col in ["Transaction Amount", "Quantity", "Customer Age", "Account Age Days"]
        if col in names(data) && any(ismissing, data[!, col])
            median_val = median(skipmissing(data[!, col]))
            replace!(data[!, col], missing => median_val)
        end
    end
    
    # ========== NEW: RISK RATIO FEATURES ==========
    if "Transaction Amount" in names(data) && "Account Age Days" in names(data)
        data.Amount_per_AccountAge = data[!, "Transaction Amount"] ./ (data[!, "Account Age Days"] .+ 1.0)
        println("  ✓ Amount/AccountAge ratio created")
    end
    
    if "Transaction Amount" in names(data)
        p95 = quantile(data[!, "Transaction Amount"], 0.95)
        p99 = quantile(data[!, "Transaction Amount"], 0.99)
        data.High_Value_Flag = [amt > p95 ? 1.0 : 0.0 for amt in data[!, "Transaction Amount"]]
        data.Very_High_Value_Flag = [amt > p99 ? 1.0 : 0.0 for amt in data[!, "Transaction Amount"]]
        println("  ✓ High value flags created")
    end
    
    # ========== NEW: QUANTITY RISK FEATURES ==========
    if "Quantity" in names(data)
        data.High_Qty_Flag = [q > 5 ? 1.0 : 0.0 for q in data[!, "Quantity"]]
        data.Very_High_Qty_Flag = [q > 10 ? 1.0 : 0.0 for q in data[!, "Quantity"]]
        
        if "Transaction Amount" in names(data)
            data.Unit_Price = data[!, "Transaction Amount"] ./ (data[!, "Quantity"] .+ 0.1)
            p95_unit = quantile(data.Unit_Price, 0.95)
            data.High_Unit_Price_Flag = [up > p95_unit ? 1.0 : 0.0 for up in data.Unit_Price]
            println("  ✓ Quantity and unit price features created")
        end
    end
    
    # ========== NEW: ACCOUNT AGE RISK ==========
    if "Account Age Days" in names(data)
        data.New_Account_Flag = [age < 30 ? 1.0 : 0.0 for age in data[!, "Account Age Days"]]
        data.Very_New_Account_Flag = [age < 7 ? 1.0 : 0.0 for age in data[!, "Account Age Days"]]
        println("  ✓ Account age flags created")
    end
    
    # ========== NEW: CUSTOMER AGE FEATURES ==========
    if "Customer Age" in names(data)
        data.Young_Customer_Flag = [age < 25 ? 1.0 : 0.0 for age in data[!, "Customer Age"]]
        data.Senior_Customer_Flag = [age > 65 ? 1.0 : 0.0 for age in data[!, "Customer Age"]]
        println("  ✓ Customer age flags created")
    end
    
    # ========== NEW: COMBINED RISK SCORE ==========
    risk_cols = []
    for col in ["High_Value_Flag", "High_Qty_Flag", "New_Account_Flag", 
                "Hour_Risk", "Is_Night", "Young_Customer_Flag"]
        if col in names(data)
            push!(risk_cols, col)
        end
    end
    
    if !isempty(risk_cols)
        data.Risk_Score = sum([data[!, col] for col in risk_cols])
        println("  ✓ Aggregated risk score created from $(length(risk_cols)) signals")
    end
    
    # ========== CLEANUP ==========
    cols_to_drop = ["Transaction ID", "Customer ID", "Transaction Date", "ParsedDate", 
                    "IP Address", "Shipping Address", "Billing Address", "Customer Location"]
    select!(data, Not(intersect(names(data), cols_to_drop)))
    
    # ========== ONE-HOT ENCODING ==========
    categorical_cols = ["Payment Method", "Product Category", "Device Used"]
    input_df = data[:, setdiff(names(data), categorical_cols)]
    
    for col in categorical_cols
        if col in names(data)
            encoded_matrix = oneHotEncoding(data[!, col])
            new_col_names = ["$(col)_$(i)" for i in 1:size(encoded_matrix, 2)]
            encoded_df = DataFrame(encoded_matrix, new_col_names)
            input_df = hcat(input_df, encoded_df)
        end
    end
    
    println("✅ Feature engineering completed!")
    return input_df
end

df_processed = preprocess_data_enhanced(df_balanced)

input_cols = setdiff(names(df_processed), [target_col])
inputs = Matrix{Float32}(df_processed[:, input_cols])
targets_bool = Bool.(vec(df_processed[:, target_col]))

println("\n📊 Dataset Info:")
println("   Features: $(size(inputs, 2))")
println("   Samples: $(size(inputs, 1))")
println("   Target balance: $(round(mean(targets_bool)*100, digits=1))% frauds")

# ============================================================================
#              TRAINING (V7.0 COMPLETE - FULLY OPTIMIZED)
# ============================================================================

println("\n" * "="^70)
println("TRAINING V7.0 COMPLETE - MAXIMUM SENSITIVITY CONFIGURATION")
println("="^70)

# ========== OPTIMIZED HYPERPARAMETERS FOR 34 FEATURES ==========
topology = [128, 64, 32]     # Deep network to leverage all features
learningRate = 0.003              # Lower for stability with deep network
maxEpochs = 800                   # More epochs for convergence
numExecutions = 1
k_folds = 3

# ========== AGGRESSIVE SETTINGS FOR HIGH SENSITIVITY ==========
posWeight = 1.8    # Era 2.0
threshold = 0.4   # Era 0.40                 # Lower threshold = more sensitive
maxEpochsVal = 25                 # More patience for validation

println("\n🔧 Configuration:")
println("   Topology: $topology")
println("   Total Parameters: ~$(256*size(inputs,2) + 128*256 + 64*128 + 32*64 + 32)")
println("   Learning Rate: $learningRate")
println("   Max Epochs: $maxEpochs")
println("   PosWeight: $posWeight (2x penalty on False Negatives)")
println("   Threshold: $threshold (Aggressive for sensitivity)")
println("   Validation Patience: $maxEpochsVal epochs")

println("\n⏱️  Training will take ~15-20 minutes with this deep network...")
println("Starting Cross-Validation...\n")

cv_indices = crossvalidation(targets_bool, k_folds)

results = ANNCrossValidationBigData(
    topology,
    (inputs, targets_bool),
    cv_indices;
    numExecutions = numExecutions,
    maxEpochs = maxEpochs,
    learningRate = learningRate,
    threshold = threshold,
    posWeight = posWeight,
    maxEpochsVal = maxEpochsVal
)

# ============================================================================
#              DETAILED RESULTS & ANALYSIS
# ============================================================================
(acc, err, sens, spec, ppv, npv, f1, global_cm) = results

println("\n" * "="^70)
println("🎯 FINAL METRICS (V7.0 COMPLETE)")
println("="^70)
println("Accuracy:    $(round(acc[1]*100, digits=2))% ± $(round(acc[2]*100, digits=2))%")
println("Sensitivity: $(round(sens[1]*100, digits=2))% ± $(round(sens[2]*100, digits=2))% ⭐ [FRAUD DETECTION]")
println("Specificity: $(round(spec[1]*100, digits=2))% ± $(round(spec[2]*100, digits=2))%")
println("Precision:   $(round(ppv[1]*100, digits=2))% ± $(round(ppv[2]*100, digits=2))%")
println("F1 Score:    $(round(f1[1]*100, digits=2))% ± $(round(f1[2]*100, digits=2))%")

TN, FP, FN, TP = global_cm[1,1], global_cm[1,2], global_cm[2,1], global_cm[2,2]
println("\n📊 CONFUSION MATRIX (Aggregated across folds):")
println("   ┌─────────────────────────────┐")
println("   │  Predicted:  NEG  │   POS   │")
println("   ├─────────────────────────────┤")
println("   │ Actual NEG:  $(lpad(Int(TN),5)) │ $(lpad(Int(FP),5))  │")
println("   │ Actual POS:  $(lpad(Int(FN),5)) │ $(lpad(Int(TP),5))  │")
println("   └─────────────────────────────┘")

total_frauds = TP + FN
detected_pct = (TP / total_frauds) * 100
missed_pct = (FN / total_frauds) * 100

println("\n🎯 FRAUD DETECTION PERFORMANCE:")
println("   ✅ Detected: $(Int(TP)) / $(Int(total_frauds)) ($(round(detected_pct, digits=1))%)")
println("   ❌ Missed:   $(Int(FN)) / $(Int(total_frauds)) ($(round(missed_pct, digits=1))%)")
println("   💰 Estimated saved: €$(Int(TP * 100)) (assuming €100/fraud)")

total_alarms = FP + TP
false_alarm_rate = total_alarms > 0 ? (FP / total_alarms * 100) : 0.0
println("\n⚠️  FALSE ALARM ANALYSIS:")
println("   Total Alerts: $(Int(total_alarms))")
println("   False Positives: $(Int(FP))")
println("   False Alarm Rate: $(round(false_alarm_rate, digits=2))%")
println("   Manual Review Cost: ~$(Int(FP * 2)) minutes (~$(round(FP * 2 / 60, digits=1)) hours)")

println("\n" * "="^70)
println("✨ V7.0 COMPLETE - Enhanced Features + Optimized Architecture")
println("="^70)

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`


✅ Packages loaded!
✅ Updated Utility functions!

Loading full dataset...
Found Total Frauds: 75060
Balanced Dataset Size: (150120, 16) (50% Fraud / 50% Legit)

🔧 Creating Enhanced Features...
  ✓ Time features created
  ✓ Amount/AccountAge ratio created
  ✓ High value flags created
  ✓ Quantity and unit price features created
  ✓ Account age flags created
  ✓ Customer age flags created
  ✓ Aggregated risk score created from 6 signals
✅ Feature engineering completed!

📊 Dataset Info:
   Features: 34
   Samples: 150120
   Target balance: 50.0% frauds

TRAINING V7.0 COMPLETE - MAXIMUM SENSITIVITY CONFIGURATION

🔧 Configuration:
   Topology: [128, 64, 32]
   Total Parameters: ~51744
   Learning Rate: 0.003
   Max Epochs: 800
   PosWeight: 1.8 (2x penalty on False Negatives)
   Threshold: 0.4 (Aggressive for sensitivity)
   Validation Patience: 25 epochs

⏱️  Training will take ~15-20 minutes with this deep network...
Starting Cross-Validation...

Params -> Threshold: 0.4 | PosWeight: 1.8

